Beautiful Soup

In [23]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [24]:
url = 'https://www.amazon.in/'
html = requests.get(url)
soup = BeautifulSoup(html.content,'html.parser') # .content gives the byte content for that html page, parser > creates the tree like structure,faster_parser :> 

In [25]:
soup

<!DOCTYPE html>

<!--[if lt IE 7]> <html lang="en-us" class="a-no-js a-lt-ie9 a-lt-ie8 a-lt-ie7"> <![endif]-->
<!--[if IE 7]>    <html lang="en-us" class="a-no-js a-lt-ie9 a-lt-ie8"> <![endif]-->
<!--[if IE 8]>    <html lang="en-us" class="a-no-js a-lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!-->
<html class="a-no-js" lang="en-us"><!--<![endif]--><head>
<meta content="text/html; charset=utf-8" http-equiv="content-type"/>
<meta charset="utf-8"/>
<meta content="IE=edge,chrome=1" http-equiv="X-UA-Compatible"/>
<title dir="ltr">Amazon.in</title>
<meta content="width=device-width" name="viewport"/>
<link href="https://images-na.ssl-images-amazon.com/images/G/01/AUIClients/AmazonUI-3c913031596ca78a3768f4e934b1cc02ce238101.secure.min._V1_.css" rel="stylesheet"/>
</head>
<body>
<!--
        To discuss automated access to Amazon data please contact api-services-support@amazon.com.
        For information about migrating to our APIs refer to our Marketplace APIs at https://developer.amazonservices.

In [26]:
soup.find('title') # finds the first occurence of any tag in that page

<title dir="ltr">Amazon.in</title>

In [27]:
soup.find_all('a') # finds every occurence of a tag in that page

[<a href="https://www.amazon.in/gp/help/customer/display.html/ref=footer_cou?ie=UTF8&amp;nodeId=200545940">Conditions of Use &amp; Sale</a>,
 <a href="https://www.amazon.in/gp/help/customer/display.html/ref=footer_privacy?ie=UTF8&amp;nodeId=200534380">Privacy Notice</a>]

In [28]:
soup.find('title').text # .string can also be used

'Amazon.in'

In [29]:
soup.find_all('a')

[<a href="https://www.amazon.in/gp/help/customer/display.html/ref=footer_cou?ie=UTF8&amp;nodeId=200545940">Conditions of Use &amp; Sale</a>,
 <a href="https://www.amazon.in/gp/help/customer/display.html/ref=footer_privacy?ie=UTF8&amp;nodeId=200534380">Privacy Notice</a>]

In [66]:
def get_data(pageno):
    url = 'https://www.amazon.in/gp/bestsellers/books/ref=zg_bs_pg'+str(pageno)+'_books?ie=UTF8&pg='+str(pageno)
    html = requests.get(url)
    #html.status_code() # to check if the url is correct
    soup = BeautifulSoup(html.content,'html.parser') # top down html parser
    data = []
    for sec in soup.find_all('div',attrs = {'class':"p13n-sc-uncoverable-faceout"}):
        book_name = sec.find('div',attrs={'class': "_cDEzb_p13n-sc-css-line-clamp-1_1Fn1y"})
        author_name = sec.find('div',attrs={'class':"a-row a-size-small"})
        rating = sec.find('span', attrs={'class':"a-icon-alt"})
        votes = sec.find('span',attrs={'class':"a-size-small"})
        price = sec.find('span',attrs={'class':"p13n-sc-price"}) # adds nan because amazon protects its price and transaction info :> to fix this remove the part before or after underscore
        book_link = sec.find('a',attrs={'class':"a-link-normal aok-block"}) 
        #print('www.amazon.in'+book_link.get('href') if book_link else 'Nan') #print a name if it exists else nan , .get :> when you want to return a property

        data.append({'Book Name': book_name.text if book_name else'Nan',
                     'Author Name': author_name.text if author_name else 'Nan',
                     'Ratings': rating.text if rating else 'Nan',
                     'Votes': votes.text if votes else 'Nan',
                     'Price': price.text if price else 'Nan',
                     'Book Link':'www.amazon.in'+book_link.get('href') if book_link else 'Nan' })

    return data


In [67]:
data = get_data(1)

In [68]:
pd.DataFrame(data)

,Book Name,Author Name,Ratings,Votes,Price,Book Link
0,My First Library Boxset For Kids (10 Board Boo...,Wonder House Books,4.5 out of 5 stars,"92,345",₹375.00,www.amazon.in/My-First-Library-Boxset-Board/dp...
1,Sanjay Arora,Sanjay Arora,Nan,Sanjay Arora,₹293.00,www.amazon.in/Brief-Case-Brands-Create-Culture...
2,PARMAR SSC Fatman | GK/GS Theory Book | Englis...,Parmar Sir,4.5 out of 5 stars,992,₹429.00,www.amazon.in/PARMAR-English-Selection-Competi...
3,The Psychology of Money,Morgan Housel,4.6 out of 5 stars,"83,559",₹289.00,www.amazon.in/Psychology-Money-Morgan-Housel/d...
4,Sonia Gandhi,Sonia Gandhi,Nan,Sonia Gandhi,"₹1,095.00",www.amazon.in/Belonging-Journey-Love-Sonia-Gan...
5,Anxious People: The No. 1 New York Times bests...,Fredrik Backman,4.6 out of 5 stars,"91,292",₹447.88,www.amazon.in/Anxious-People-bestseller-author...
6,"Atomic Habits: Tiny Changes, Remarkable Results",James Clear,4.6 out of 5 stars,"112,894",₹469.00,www.amazon.in/Atomic-Habits-James-Clear/dp/184...
7,White Nights by Fyodor Dostoyevsky: A Timeless...,Fyodor Dostoyevsky,4.6 out of 5 stars,"5,788",₹105.00,www.amazon.in/White-Nights-Penguin-Little-Clas...
8,The Mountain Is You: Transforming Self-Sabotag...,Brianna Wiest,4.5 out of 5 stars,"28,492",₹266.00,www.amazon.in/Mountain-You-Transforming-Self-S...
9,TUESDAYS WITH MORRIE (EXPORT) (A FORMAT),Mitch Albom,4.6 out of 5 stars,"38,289",₹214.00,www.amazon.in/TUESDAYS-MORRIE-EXPORT-FORMAT-Mi...


In [70]:
number_of_pages = 2
all_data = []

for pageno in range(1,number_of_pages+1):
    all_data.extend(get_data(pageno)) # Extend :> it will keep on adding them as elemets of the same list, .append :> will create a list of lists here

In [71]:
df = pd.DataFrame(all_data)

In [72]:
df

,Book Name,Author Name,Ratings,Votes,Price,Book Link
0,My First Library Boxset For Kids (10 Board Boo...,Wonder House Books,4.5 out of 5 stars,"92,345",₹375.00,www.amazon.in/My-First-Library-Boxset-Board/dp...
1,Sanjay Arora,Sanjay Arora,Nan,Sanjay Arora,₹293.00,www.amazon.in/Brief-Case-Brands-Create-Culture...
2,PARMAR SSC Fatman | GK/GS Theory Book | Englis...,Parmar Sir,4.5 out of 5 stars,992,₹429.00,www.amazon.in/PARMAR-English-Selection-Competi...
3,The Psychology of Money,Morgan Housel,4.6 out of 5 stars,"83,559",₹289.00,www.amazon.in/Psychology-Money-Morgan-Housel/d...
4,Sonia Gandhi,Sonia Gandhi,Nan,Sonia Gandhi,"₹1,095.00",www.amazon.in/Belonging-Journey-Love-Sonia-Gan...
...,...,...,...,...,...,...
95,Mathematics at Advance Level 9 CBSE Textbook N...,Tovira,4.4 out of 5 stars,Tovira,₹120.00,www.amazon.in/Mathematics-Advance-Textbook-gra...
96,PW Med Easy Zoology 2.0 and Botany Handwritten...,Dr. Vipin Sharma,4.6 out of 5 stars,597,₹795.00,www.amazon.in/Handwritten-Flowcharts-Mnemonics...
97,Rich Dad Poor Dad: What the Rich Teach Their K...,Robert T. Kiyosaki,4.5 out of 5 stars,"38,305",₹370.00,www.amazon.in/Rich-Dad-Poor-Middle-Anniversary...
98,"The Forty Rules of Love: A Journey of Love, Fa...",Elif Shafak,4.5 out of 5 stars,"20,902",₹423.00,www.amazon.in/Forty-Rules-Love-Elif-Shafak/dp/...


In [ ]:
#Publisher, pub date, top 3 text reviews for a book